# Dashboard: allocating undecided 2023 PSOE voters

Run with `uv run python main.py app`. It only loads what `train.ipynb` saved in `artifacts/`: nothing is fitted
here, and the saved preprocessor is only used with `.transform`.

Four tabs:
1. **Threshold**: move the threshold and watch the confusion matrix and the error of the estimate change.
2. **Three implementations**: scikit-learn, the manual PyTorch loop and nn.Module against the baselines on the
   test months: predicted vs actual, where the coefficients differ, and convergence.
3. **Data**: churn and the undecided over time, where leavers go, churn by feature, and how the undecided differ.
4. **One voter**: one hypothetical undecided voter through the three saved models.

## Load artifacts

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import gradio as gr
import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch

import features as F
from models import LogisticNet, manual_pytorch_proba, monthly_abs_error, monthly_errors, nn_module_proba

ROOT = Path.cwd()  # main.py runs this with cwd = repo root
ARTIFACTS = ROOT / "artifacts"
if not (ARTIFACTS / "metrics.json").exists():
    raise SystemExit("artifacts/ is empty: run `uv run python main.py train` first.")

metrics = json.loads((ARTIFACTS / "metrics.json").read_text())
predictions = pd.read_csv(ARTIFACTS / "predictions.csv")
preprocessor = joblib.load(ARTIFACTS / "preprocessor.joblib")
segmenter = joblib.load(ARTIFACTS / "segmenter.joblib")
sk_model = joblib.load(ARTIFACTS / "sklearn_logreg.joblib")
manual = torch.load(ARTIFACTS / "manual_pytorch.pt")
nn_model = LogisticNet(len(metrics["feature_names"]))
nn_model.load_state_dict(torch.load(ARTIFACTS / "nn_module.pt"))

T_VAL = metrics["thresholds"]["sklearn"]  # chosen on validation, never on test
MODELS = {"sklearn": "scikit-learn", "manual_pytorch": "Manual PyTorch loop", "nn_module": "nn.Module + optim"}
TEST = predictions[predictions["set"] == "test"].reset_index(drop=True)
CMP = metrics["comparison_test"]
AGREE = metrics["agreement_test"]

/Users/jimenasanchezcurto/Downloads/daiist-assignment-1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data for the descriptive charts

`features.engineer` has nothing fitted in it, so this is the same view of the data the models were trained on.
The target only exists for voters who declared an intention.

In [2]:
raw = F.load_raw()
data = F.engineer(raw)
data["split"] = F.split_of(raw["study"])
data["declared"] = F.is_declared(raw)
for col in ("churn", "weight", "study", "destination", "field_date"):
    data[col] = raw[col]
DECLARED = data[data["declared"]]
month_of = raw.groupby("study")["field_date"].min().str[:7].to_dict()
MONTH_LABEL = {s: pd.Timestamp(m + "-01").strftime("%b %Y") for s, m in month_of.items()}

DESTINATION_EN = {
    "En blanco": "Blank vote", "No votaría": "Would not vote", "Otro partido": "Other party",
    "Voto nulo": "Null vote", "VOX": "Vox",
}

## Chart style

White page, one typeface, no titles or legends inside the charts. Each chart's title and caption are HTML above
it, and the caption names the series with a coloured key (`key()`), or the line is labelled at its end. Blue
always means "stays PSOE" and orange "leaves". The three implementations also get different dash styles,
because their lines lie on top of each other.

In [3]:
PAGE = "#ffffff"
INK, INK_2, MUTED, FAINT = "#1b1b1b", "#474747", "#707070", "#a8a8a8"
GRID, RULE = "#efefef", "#dddddd"
BLUE, ORANGE, AQUA = "#2a78d6", "#e0612f", "#1baf7a"
STAY, CHURN = BLUE, ORANGE
MODEL_STYLE = {"sklearn": (BLUE, "solid"), "manual_pytorch": (ORANGE, "dash"), "nn_module": (AQUA, "dot")}
FONT = "-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif"


def style(fig: go.Figure, x: str | None = None, y: str | None = None, height: int = 300,
          right: int = 0) -> go.Figure:
    """Shared layout: transparent, no title, no legend. Series are named in the text above the chart
    (see `key`) or labelled at the end of their line, so the chart reads as part of the page."""
    fig.update_layout(
        height=height, template="none", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        font=dict(family=FONT, size=12, color=MUTED), margin=dict(l=0, r=right, t=10, b=0),
        showlegend=False, bargap=0.25,
        hoverlabel=dict(bgcolor=PAGE, bordercolor=RULE, font=dict(family=FONT, size=12, color=INK)),
    )
    axis = dict(zeroline=False, ticks="", automargin=True, tickfont=dict(size=11, color=MUTED),
                title=dict(font=dict(size=12, color=MUTED), standoff=10))
    fig.update_xaxes(title_text=x, showgrid=False, showline=True, linecolor=RULE, **axis)
    fig.update_yaxes(title_text=y, gridcolor=GRID, showline=False, **axis)
    return fig


def note(fig: go.Figure, x, y, text: str, color: str = MUTED, xanchor: str = "left", yanchor: str = "middle",
         xshift: int = 6, yshift: int = 0, size: int = 11, **ref) -> None:
    fig.add_annotation(x=x, y=y, text=text, showarrow=False, xanchor=xanchor, yanchor=yanchor, xshift=xshift,
                       yshift=yshift, font=dict(size=size, color=color), **ref)


def key(text: str, color: str, mark: str = "box") -> str:
    """A series name in running text, with its colour swatch: replaces the chart legend.
    mark: box (bars), line, dash, dot (line styles), point (markers)."""
    return f'<span class="key {mark}" style="--c:{color}">{text}</span>'


def pct(v: float, d: int = 0) -> str:
    return f"{v:.{d}%}"


def num(v, fmt: str) -> str:
    """Format a metric that may be missing (None, or NaN after the JSON round trip)."""
    return "" if v is None or (isinstance(v, float) and np.isnan(v)) else format(v, fmt)


def sci(v: float) -> str:
    """2.6e-06 -> 2.6 × 10⁻⁶"""
    mantissa, exp = f"{v:.1e}".split("e")
    return f"{mantissa} × 10<sup>{int(exp)}</sup>".replace("-", "−")


def table(header: list[str], rows: list[list[str]], cls: str = "") -> str:
    head = "".join(f"<th>{h}</th>" for h in header)
    body = "".join("<tr>" + "".join(f"<td>{c}</td>" for c in r) + "</tr>" for r in rows)
    return f'<table class="tbl {cls}"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'

## Header

In [4]:
n_declared = sum(s["rows"] for s in metrics["split"].values())
header_html = f"""
<div class="page-head">
  <h1>Undecided PSOE voters</h1>
  <p>Every month, about one in seven people who voted PSOE in 2023 tell the CIS they don't know yet who they'll
  vote for. A poll still has to count them somewhere. This model learns from the 2023 PSOE voters who did answer
  and decides, for each undecided one, whether they go back to PSOE or not.</p>
  <p class="meta">CIS monthly barometers, April 2025 to September 2026: {n_declared:,} voters who answered,
  {metrics["undecided_rows"]:,} who didn't. Each barometer re-weighted to the 2023 result and the census.</p>
</div>"""

## Tab 1: threshold and error

Everything here reads the **test months** (May to September 2026). The declared voters of those months are
allocated as if they were undecided, which is the only way to check an allocation against the truth. The
business number is the mean absolute monthly error of the estimated churn rate, in percentage points.

In [5]:
Y_TEST = TEST["churn"].to_numpy()
P_TEST = TEST["p_sklearn"].to_numpy()
M_TEST = TEST["study"].to_numpy()
W_TEST = TEST["weight"].to_numpy()  # corrected survey weight (data/raking.py)
RULE_TEST = (TEST["closest_group"] != "psoe").to_numpy()
THRESHOLDS = np.round(np.arange(0.01, 1.0, 0.01), 2)
ERROR_CURVE = np.array([monthly_abs_error(Y_TEST, P_TEST > t, M_TEST, W_TEST) for t in THRESHOLDS])
REFERENCES = [  # estimates that need no threshold, for comparison
    ("Closest-party rule", CMP["closest_party_rule"]["monthly_abs_error_pp"]),
    ("Training churn rate", CMP["naive_constant"]["monthly_abs_error_pp"]),
    ("Average probability", CMP["sklearn_expected"]["monthly_abs_error_pp"]),
]


def summary_html(t: float) -> str:
    y, a = Y_TEST, P_TEST > t
    fp, fn = int((a & (y == 0)).sum()), int((~a & (y == 1)).sum())
    stats = [
        (f"{monthly_abs_error(y, a, M_TEST, W_TEST):.1f} pp", "average monthly error"),
        (pct(np.average(a, weights=W_TEST), 1), f"estimated churn (actual {pct(np.average(y, weights=W_TEST), 1)})"),
        (f"{fp:,} / {fn:,}", "false positives / negatives"),
        (pct((a == (y == 1)).mean(), 1), "voters allocated correctly"),
    ]
    cells = "".join(f"<div><b>{v}</b><span>{label}</span></div>" for v, label in stats)
    return f'<div class="stats">{cells}</div>'


def confusion_html(t: float) -> str:
    y, a = Y_TEST, P_TEST > t
    n = len(y)
    tn, fp = int((~a & (y == 0)).sum()), int((a & (y == 0)).sum())
    fn, tp = int((~a & (y == 1)).sum()), int((a & (y == 1)).sum())

    def cell(count: int, what: str, wrong: bool) -> str:
        return (f'<div class="cell{" wrong" if wrong else ""}"><b>{count:,}</b>'
                f'<span>{what}, {count / n:.1%}</span></div>')

    return (
        '<div class="cm"><div></div><div class="colhead">Allocated to PSOE</div>'
        '<div class="colhead">Allocated elsewhere</div>'
        f'<div class="rowhead">Stayed</div>{cell(tn, "correct", False)}{cell(fp, "false positive", True)}'
        f'<div class="rowhead">Left</div>{cell(fn, "false negative", True)}{cell(tp, "correct", False)}</div>'
        f'<p class="cap">The poll only publishes a count, so the two kinds of error offset each other: '
        f'{fp:,} − {fn:,} = {fp - fn:+,} interviews off in total.</p>'
    )


def error_curve_fig(t: float) -> go.Figure:
    fig = go.Figure()
    for name, val in REFERENCES:
        fig.add_hline(y=val, line=dict(color=FAINT, width=1, dash="dot"))
        note(fig, 1, val, f"{name} {val:.1f} pp", xanchor="right", yanchor="bottom", xshift=0, yshift=1,
             xref="paper")
    fig.add_scatter(x=THRESHOLDS, y=ERROR_CURVE, mode="lines", line=dict(color=BLUE, width=2.5),
                    hovertemplate="threshold %{x:.2f}<br><b>%{y:.1f} pp</b><extra></extra>")
    fig.add_vline(x=T_VAL, line=dict(color=RULE, width=1))
    note(fig, T_VAL, 1, f"chosen on validation: {T_VAL:.2f}", yanchor="top", yref="paper")
    err = monthly_abs_error(Y_TEST, P_TEST > t, M_TEST, W_TEST)
    fig.add_scatter(x=[t], y=[err], mode="markers", hoverinfo="skip",
                    marker=dict(color=INK, size=10, line=dict(color=PAGE, width=2)))
    fig.update_xaxes(range=[0, 1], tickvals=[0.2, 0.4, 0.6, 0.8, 1.0], tickformat=".1f")
    fig.update_yaxes(range=[0, max(v for _, v in REFERENCES) * 1.6], ticksuffix=" pp")
    return style(fig)


def monthly_fig(t: float) -> go.Figure:
    rows = []
    for study in np.unique(M_TEST):
        m = M_TEST == study
        w = W_TEST[m]
        rows.append((MONTH_LABEL[study], np.average(Y_TEST[m], weights=w), np.average(P_TEST[m] > t, weights=w),
                     np.average(RULE_TEST[m], weights=w)))
    months, actual, model, rule = zip(*rows)
    fig = go.Figure()
    fig.add_scatter(x=months, y=rule, mode="markers", marker=dict(color=FAINT, size=10, symbol="diamond"),
                    hovertemplate="%{x}<br>rule %{y:.1%}<extra></extra>")
    fig.add_scatter(x=months, y=model, mode="markers", marker=dict(color=BLUE, size=11),
                    hovertemplate="%{x}<br>model %{y:.1%}<extra></extra>")
    fig.add_scatter(x=months, y=actual, mode="markers",
                    marker=dict(color=INK, size=16, symbol="line-ew-open", line=dict(width=2.5)),
                    hovertemplate="%{x}<br>actual %{y:.1%}<extra></extra>")
    fig.update_yaxes(tickformat=".0%", rangemode="tozero")
    return style(fig)


def score_split_fig(t: float) -> go.Figure:
    fig = go.Figure()
    for cls, name, color in ((0, "stayed", STAY), (1, "left", CHURN)):
        fig.add_histogram(x=P_TEST[Y_TEST == cls], histnorm="percent", xbins=dict(start=0, end=1, size=0.025),
                          marker=dict(color=color), opacity=0.7,
                          hovertemplate=f"p about %{{x:.2f}}<br>%{{y:.1f}}% of voters who {name}<extra></extra>")
    fig.add_vline(x=t, line=dict(color=INK, width=1.5))
    fig.update_layout(barmode="overlay")
    fig.update_xaxes(range=[0, 1], tickformat=".1f")
    fig.update_yaxes(ticksuffix="%")
    return style(fig)


def decide(t):
    return summary_html(t), error_curve_fig(t), confusion_html(t), monthly_fig(t), score_split_fig(t)

## Tab 2: three implementations

The brief's predicted-vs-actual plot is the calibration curve: of the voters the model gives 40%, do 40% leave?
It matters here because the estimate is built from the probabilities.

The coefficient chart shows *where* PyTorch and scikit-learn differ, not just that they are close: the
difference per parameter, added up within each one-hot block.

In [6]:
COMPARISON_ROWS = [
    ("sklearn", "scikit-learn"), ("manual_pytorch", "Manual PyTorch loop"), ("nn_module", "nn.Module + optim"),
    ("sklearn_expected", "Average probability, no threshold"), ("naive_constant", "Training churn rate"),
    ("closest_party_rule", "Closest party is PSOE"), ("all_return", "All undecided go back to PSOE"),
]


def comparison_html() -> str:
    rows = []
    for key_, name in COMPARISON_ROWS:
        r = CMP[key_]
        rows.append([name, num(r["auc"], ".3f"), num(r["logloss"], ".4f"), num(r["threshold"], ".2f"),
                     pct(r["estimated_churn"], 1), f"{r['monthly_abs_error_pp']:.2f} pp", num(r["accuracy"], ".1%")])
    return table(["", "AUC", "Log-loss", "Threshold", "Estimated churn", "Monthly error", "Accuracy"], rows,
                 cls="cmp")


def comparison_caption() -> str:
    gap = AGREE["sklearn_vs_manual_pytorch"]["max_abs_dprob"]
    ci = metrics["test_ci"]
    return (f'<p class="cap">Actual churn in the test months: {pct(CMP["sklearn"]["actual_churn"], 1)}. '
            f'The three implementations never differ by more than {sci(gap)} in a predicted probability, and '
            f'they allocate every voter the same way.<br>95% bootstrap intervals: AUC {ci["auc_ci"][0]:.3f} to '
            f'{ci["auc_ci"][1]:.3f}; monthly error {ci["model_error_ci"][0]:.2f} to {ci["model_error_ci"][1]:.2f} '
            f'pp for the model and {ci["rule_error_ci"][0]:.2f} to {ci["rule_error_ci"][1]:.2f} pp for the '
            f'closest-party rule.</p>')


def months_html() -> str:
    rows = [[MONTH_LABEL[r["study"]], f"{r['rows']:,}", pct(r["actual_churn"], 1), pct(r["model"], 1),
             pct(r["model_expected"], 1), pct(r["rule"], 1), f"{r['auc']:.3f}"]
            for r in metrics["per_month_test"]]
    return table(["", "Voters", "Actual", "Model", "Avg. prob.", "Rule", "AUC"], rows)


def calibration_fig() -> go.Figure:
    fig = go.Figure()
    fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", hoverinfo="skip", line=dict(color=FAINT, width=1, dash="dot"))
    for m, name in MODELS.items():
        color, dash = MODEL_STYLE[m]
        g = TEST.groupby(pd.qcut(TEST[f"p_{m}"], 10, duplicates="drop"), observed=True).agg(
            pred=(f"p_{m}", "mean"), actual=("churn", "mean"), n=("churn", "size"))
        fig.add_scatter(x=g["pred"], y=g["actual"], customdata=g["n"], mode="lines+markers",
                        line=dict(color=color, dash=dash, width=2), marker=dict(size=7 if m == "sklearn" else 5),
                        hovertemplate="predicted %{x:.1%}<br>actual %{y:.1%}<br>%{customdata} voters"
                                      f"<extra>{name}</extra>")
    fig.update_xaxes(range=[0, 1.02], tickformat=".0%")
    fig.update_yaxes(range=[0, 1.02], tickformat=".0%")
    return style(fig, "Predicted probability of leaving", "Share who actually left", height=340)


ONE_HOT_BLOCKS = {
    "closest_group": "Closest party", "age_band": "Age band", "education": "Education", "religion": "Religion",
    "work_status": "Work status", "social_class": "Social class", "region": "Region",
    "marital_status": "Marital status", "segment": "Segment",
}


def coef_gap_fig() -> go.Figure:
    """PyTorch minus scikit-learn, per parameter, summed within each one-hot block."""
    names = ["intercept"] + metrics["feature_names"]
    d = pd.Series(np.subtract(metrics["coefficients"]["manual_pytorch"], metrics["coefficients"]["sklearn"]),
                  index=names)
    rows = {"Intercept": d["intercept"]}
    for col, label in ONE_HOT_BLOCKS.items():
        block = d[d.index.str.startswith(f"categorical__{col}_")]
        if len(block):
            rows[f"{label} ({len(block)})"] = block.sum()
    rest = d[~d.index.str.startswith("categorical__") & (d.index != "intercept")]
    rows[f"Largest of the other {len(rest)}"] = rest.abs().max()
    s = pd.Series(rows) * 1e5
    fig = go.Figure(go.Bar(
        x=s.values, y=s.index, orientation="h", marker=dict(color=[INK_2] + [FAINT] * (len(s) - 1)),
        text=[f"{v:+.1f}" for v in s.values], textposition="outside", cliponaxis=False,
        textfont=dict(size=11, color=INK_2), hovertemplate="%{y}<br>%{x:+.2f} × 10⁻⁵<extra></extra>"))
    fig.add_vline(x=0, line=dict(color=RULE, width=1))
    pad = s.abs().max() * 0.25
    fig.update_xaxes(range=[s.min() - pad, s.max() + pad], showline=False, ticksuffix="")
    fig.update_yaxes(autorange="reversed", showgrid=False, tickfont=dict(size=12, color=INK_2), ticksuffix="  ")
    return style(fig, None, None, height=340)


def convergence_fig() -> go.Figure:
    hp = metrics["hyperparameters"]
    fig = go.Figure()
    for m in ("manual_pytorch", "nn_module"):
        color, dash = MODEL_STYLE[m]
        hist = np.array(metrics["loss_history"][m])
        gap = np.clip(hist - hp["sklearn_train_objective"], 1e-14, None)
        fig.add_scatter(x=np.arange(len(hist)) * 50, y=gap, mode="lines", line=dict(color=color, dash=dash, width=2),
                        hovertemplate="epoch %{x}<br>gap %{y:.1e}<extra>" + MODELS[m] + "</extra>")
    fig.update_yaxes(type="log", exponentformat="power", dtick=2, range=[-14.3, 0.3])
    return style(fig, "Epoch", None, height=300)

## Tab 3: data

Feature and target distributions, the undecided over time, and how they compare with the voters the model was
trained on. Population figures use the CIS weight; the models do not.

In [7]:
SPLIT_SHADE = {"val": "#f3f6fb", "test": "#fbf3ef"}


def shade_splits(fig: go.Figure, months: pd.Series, split: pd.Series) -> None:
    for s, label in (("train", "train"), ("val", "validation"), ("test", "test")):
        mm = months[split == s]
        if s in SPLIT_SHADE:
            fig.add_vrect(x0=mm.min() - pd.Timedelta(days=14), x1=mm.max() + pd.Timedelta(days=14),
                          fillcolor=SPLIT_SHADE[s], line_width=0, layer="below")
        note(fig, mm.min() - pd.Timedelta(days=12), 1, label, yanchor="top", xshift=0, yref="paper")


def end_label(fig: go.Figure, x, y, text: str, color: str) -> None:
    """Name a line at its last point instead of in a legend."""
    note(fig, x, y, text, color=color, xshift=8, size=12)


def over_time_fig() -> go.Figure:
    g = data.groupby("study").apply(lambda x: pd.Series({
        "churn": np.average(x.loc[x["declared"], "churn"], weights=x.loc[x["declared"], "weight"]),
        "undecided": np.average(~x["declared"], weights=x["weight"]),
        "split": x["split"].iloc[0]}), include_groups=False).reset_index()
    g["month"] = pd.to_datetime(g["study"].map(month_of) + "-15")
    fig = go.Figure()
    shade_splits(fig, g["month"], g["split"])
    fig.add_scatter(x=g["month"], y=g["churn"], mode="lines+markers", line=dict(color=CHURN, width=2.5),
                    marker=dict(size=6), hovertemplate="%{x|%b %Y}<br>churn %{y:.1%}<extra></extra>")
    fig.add_scatter(x=g["month"], y=g["undecided"], mode="lines+markers", line=dict(color=INK_2, width=1.5, dash="dot"),
                    marker=dict(size=5), hovertemplate="%{x|%b %Y}<br>undecided %{y:.1%}<extra></extra>")
    end_label(fig, g["month"].iloc[-1], g["churn"].iloc[-1], "churn", CHURN)
    end_label(fig, g["month"].iloc[-1], g["undecided"].iloc[-1], "undecided", INK_2)
    fig.update_yaxes(tickformat=".0%", rangemode="tozero")
    fig.update_xaxes(tickformat="%b<br>%Y", dtick="M2",
                     range=[g["month"].min() - pd.Timedelta(days=20), g["month"].max() + pd.Timedelta(days=20)])
    return style(fig, right=80)


def correction_fig() -> go.Figure:
    """Share of the sample that recalls voting PSOE in 2023, with the CIS weight and after the correction."""
    r = pd.DataFrame([{"study": int(s), "before": d["psoe_recall_share_before"], "after": d["psoe_recall_share_after"]}
                      for s, d in metrics["raking"].items()])
    r["month"] = pd.to_datetime(r["study"].map(month_of) + "-15")
    official = r["after"].iloc[0]
    fig = go.Figure()
    fig.add_hline(y=official, line=dict(color=INK_2, width=1.5))
    note(fig, 1, official, f"23J result {official:.1f}%", color=INK_2, xanchor="right", yanchor="top", xshift=0,
         yshift=-2, xref="paper")
    fig.add_scatter(x=r["month"], y=r["before"], mode="lines+markers", line=dict(color=CHURN, width=2.5),
                    marker=dict(size=6), hovertemplate="%{x|%b %Y}<br>%{y:.1f}% with the CIS weight<extra></extra>")
    fig.update_yaxes(ticksuffix="%", range=[25, max(r["before"]) + 3])
    fig.update_xaxes(tickformat="%b<br>%Y", dtick="M2")
    return style(fig, height=260)


def correction_html() -> str:
    r = metrics["raking"].values()
    before = np.mean([d["psoe_recall_share_before"] for d in r])
    deff = np.mean([d["deff"] for d in r])
    factor = np.mean([d["psoe_factor_mean"] for d in r])
    stats = [(f"{before:.1f}%", "recall PSOE with the CIS weight"), (f"{factor:.2f}", "average weight change, PSOE voters"),
             (f"{deff:.1f}", "design effect")]
    return '<div class="stats three">' + "".join(f"<div><b>{v}</b><span>{l}</span></div>" for v, l in stats) + "</div>"


def allocation_fig() -> go.Figure:
    a = pd.DataFrame(metrics["allocation"])
    a["month"] = pd.to_datetime(a["study"].map(month_of) + "-15")
    fig = go.Figure()
    for col, name, color, dash in (("retention_all_return", "all go back", FAINT, "dot"),
                                   ("retention_rule", "closest-party rule", INK_2, "dash"),
                                   ("retention_model", "model", BLUE, "solid")):
        fig.add_scatter(x=a["month"], y=a[col], mode="lines", line=dict(color=color, dash=dash, width=2),
                        hovertemplate="%{x|%b %Y}<br>" + name + " %{y:.1%}<extra></extra>")
    fig.update_yaxes(tickformat=".0%")
    fig.update_xaxes(tickformat="%b<br>%Y", dtick="M2")
    return style(fig)


def destination_fig() -> go.Figure:
    ch = DECLARED[DECLARED["churn"] == 1].assign(dest=lambda d: d["destination"].replace(DESTINATION_EN))
    share = ch.groupby("dest")["weight"].sum() / ch["weight"].sum()
    top = share.nlargest(7)
    share = pd.concat([top, pd.Series({"Everything else": share.drop(top.index).sum()})]).iloc[::-1]
    fig = go.Figure(go.Bar(
        x=share.values, y=share.index, orientation="h",
        marker=dict(color=[FAINT if i == "Everything else" else CHURN for i in share.index]),
        text=[f"{v:.0%}" for v in share.values], textposition="outside", cliponaxis=False,
        textfont=dict(color=INK_2, size=11), hovertemplate="%{y}: %{x:.1%} of leavers<extra></extra>"))
    fig.update_xaxes(visible=False, range=[0, share.max() * 1.2])
    fig.update_yaxes(showgrid=False, tickfont=dict(color=INK_2, size=12), ticksuffix="  ")
    return style(fig)


PROFILE_LABELS = {
    "rate_sanchez": ("Rating of Sánchez (1 to 10)", ".1f"), "gap_best_rival": ("Sánchez minus best rival", "+.1f"),
    "rate_feijoo": ("Rating of Feijóo", ".1f"), "ideology": ("Ideology (1 left, 10 right)", ".1f"),
    "probvoto": ("Probability of voting (0 to 10)", ".1f"), "eco_spain": ("Spain's economy (1 to 5)", ".1f"),
    "closest_party_psoe": ("Closest party is PSOE", ".0%"),
}


def undecided_html() -> str:
    prof = metrics["undecided_profile"]
    rows = [[label, format(prof[k]["declared"], f), format(prof[k]["undecided"], f)]
            for k, (label, f) in PROFILE_LABELS.items() if k in prof]
    return table(["", "Answered", "Undecided"], rows)


DRIVERS = {
    "Rating of Sánchez": "rate_sanchez", "Sánchez minus best rival": "gap_best_rival",
    "Closest party": "closest_group", "Probability of voting": "probvoto", "Ideology": "ideology",
    "Spain's economy": "eco_spain", "Age": "age_band",
}
LEVEL_EN = {"psoe": "PSOE", "left": "Left", "regional": "Regional", "right": "Right", "none": "None",
            "no_answer": "No answer", "other": "Other"}
LEVEL_ORDER = {"closest_group": ["PSOE", "Left", "Regional", "Right", "None", "Other", "No answer"]}


def driver_levels(col: str) -> tuple[pd.Series, list]:
    v = DECLARED[col]
    if col == "gap_best_rival":
        v = v.clip(-5, 5)
    numeric = v.dtype.kind == "f"
    v = v.astype(object).map(lambda x: "No answer" if pd.isna(x) else int(x) if numeric else LEVEL_EN.get(x, x))
    levels = sorted(v.unique(), key=lambda x: (x == "No answer", x if isinstance(x, int) else 0, str(x)))
    if col in LEVEL_ORDER:
        levels = [lv for lv in LEVEL_ORDER[col] if lv in levels]
    return v, levels


def driver_figs(label: str) -> tuple[go.Figure, go.Figure]:
    col = DRIVERS[label]
    v, levels = driver_levels(col)
    w, y = DECLARED["weight"], DECLARED["churn"]
    xs = [f"{lv:+d}" if col == "gap_best_rival" and isinstance(lv, int) else str(lv) for lv in levels]

    g = pd.DataFrame({"v": v, "w": w, "y": y}).groupby("v").apply(
        lambda d: pd.Series({"rate": np.average(d["y"], weights=d["w"]), "n": len(d)}),
        include_groups=False).reindex(levels)
    avg = np.average(y, weights=w)
    rate = go.Figure(go.Bar(x=xs, y=g["rate"], customdata=g["n"],
                            marker=dict(color=[FAINT if x == "No answer" else CHURN for x in xs]),
                            hovertemplate="%{x}<br><b>%{y:.0%}</b> leave<br>%{customdata:,} voters<extra></extra>"))
    rate.add_hline(y=avg, line=dict(color=INK_2, width=1, dash="dot"))
    note(rate, 1, avg, f"average {avg:.0%}", color=INK_2, xanchor="right", yanchor="bottom", xshift=0,
         xref="paper")
    rate.update_yaxes(tickformat=".0%", range=[0, 1])
    rate.update_xaxes(type="category")
    style(rate)

    dist = go.Figure()
    for cls, name, color in ((0, "stayed", STAY), (1, "left", CHURN)):
        m = y == cls
        share = (w[m].groupby(v[m]).sum() / w[m].sum()).reindex(levels, fill_value=0)
        dist.add_bar(x=xs, y=share.values, marker=dict(color=color),
                     hovertemplate="%{x}<br>%{y:.1%} of voters who " + name + "<extra></extra>")
    dist.update_layout(barmode="group", bargroupgap=0.08)
    dist.update_yaxes(tickformat=".0%")
    dist.update_xaxes(type="category")
    style(dist)
    return rate, dist

## Tab 4: one voter

One hypothetical undecided 2023 PSOE voter through the saved preprocessor and all three models. The bar chart
breaks the scikit-learn score down exactly: intercept plus coefficient times feature value gives the log-odds.

In [8]:
codebook = json.loads(F.CODEBOOK.read_text())


def code_choices(var: str, keep: range) -> dict[str, int]:
    return {lab.replace("(NO LEER) ", ""): int(k) for k, lab in codebook[f"{var}__values"].items() if int(k) in keep}


ECO = {"Very good": 1, "Good": 2, "Fair": 3, "Bad": 4, "Very bad": 5, "Doesn't know": 8}  # CIS codes for ECOESP/ECOPER
INCOME = code_choices("INGRESHOG", range(1, 7)) | {"Doesn't know / no answer": 9}
TOWN = code_choices("TAMUNI", range(1, 8))
PROBLEMS = {"(none)": None} | {lab: int(k) for k, lab in sorted(codebook["PROBLEM__values"].items(),
                                                                   key=lambda kv: kv[1])}
train_raw = raw[(data["split"] == "train") & data["declared"]]
NOMINAL_INPUTS = {c: train_raw[c].value_counts().index.tolist()
                  for c in ("CERCANIA", "SEXO", "ESTUDIOS", "RELIGION", "SITLAB", "CLASESOCIAL", "CCAA", "ECIVIL")}
mode_of = {c: v[0] for c, v in NOMINAL_INPUTS.items()}

FEATURE_EN = {
    "rate_sanchez": "Rating of Sánchez", "rate_feijoo": "Rating of Feijóo", "rate_diaz": "Rating of Díaz",
    "rate_abascal": "Rating of Abascal", "gap_best_rival": "Sánchez minus best rival",
    "sanchez_top": "Sánchez rated top", "probvoto": "Probability of voting", "ideology": "Ideology",
    "eco_spain": "Spain's economy", "eco_personal": "Personal economy", "income_rank": "Income",
    "income_dk": "Income: no answer", "town_size": "Town size", "female": "Woman",
}


def pretty(feature: str) -> str:
    name = feature.split("__", 1)[1]
    if name in FEATURE_EN:
        return FEATURE_EN[name]
    if name.startswith("problem_"):
        return "Problem: " + name[8:].replace("_", " ")
    if name.startswith("closest_group_"):
        return "Closest party: " + LEVEL_EN.get(name[14:], name[14:])
    for prefix in ("religion", "social_class", "education", "work_status", "region", "marital_status", "age_band",
                   "segment"):
        if name.startswith(prefix + "_"):
            return f"{prefix.replace('_', ' ').capitalize()}: {name[len(prefix) + 1:].replace('_', ' ')}"
    return name.replace("_", " ")


def score_voter(closest, sanchez, feijoo, diaz, abascal, probvoto, ideology, eco_spain, eco_personal,
                p1, p2, p3, age, sex, education, religion, work, social_class, region, marital, income, town):
    row = pd.DataFrame([{
        "CERCANIA": closest,
        "VALORALIDERES_1": sanchez, "VALORALIDERES_2": feijoo, "VALORALIDERES_3": diaz, "VALORALIDERES_4": abascal,
        "PROBVOTO": probvoto, "ESCIDEOL": ideology, "ECOESP": ECO[eco_spain], "ECOPER": ECO[eco_personal],
        "PESPANNA1": PROBLEMS[p1], "PESPANNA2": PROBLEMS[p2], "PESPANNA3": PROBLEMS[p3],
        "EDAD": age, "SEXO": sex, "ESTUDIOS": education, "RELIGION": religion, "SITLAB": work,
        "CLASESOCIAL": social_class, "CCAA": region, "ECIVIL": marital,
        "INGRESHOG": INCOME[income], "TAMUNI": TOWN[town],
    }])
    e = F.engineer(row)
    if "segment" in metrics["selected_blocks"]:
        e["segment"] = "S" + str(segmenter.predict(e[F.SEGMENT_INPUTS])[0])
    X = preprocessor.transform(e)
    probs = {
        "sklearn": float(sk_model.predict_proba(X)[0, 1]),
        "manual_pytorch": float(manual_pytorch_proba(X, manual["w"], manual["b"])[0]),
        "nn_module": float(nn_module_proba(X, nn_model)[0]),
    }
    p = probs["sklearn"]
    side = "above" if p > T_VAL else "below"
    verdict = "somewhere other than PSOE" if p > T_VAL else "back to PSOE"
    impl = table(["", "p(leaves)"], [[MODELS[m], f"{v:.6f}"] for m, v in probs.items()])
    card = (f'<div class="score"><p class="big">{p:.0%}</p>'
            f'<p>chance this voter doesn\'t go back to PSOE. That is {side} the {T_VAL:.2f} threshold, so the poll '
            f'counts them <b>{verdict}</b>.</p>{impl}</div>')

    contrib = pd.Series(X[0] * sk_model.coef_.ravel(), index=metrics["feature_names"])
    contrib = contrib[contrib.abs() > 1e-9]
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).index[:9])
    rest = contrib.drop(top.index).sum()
    intercept = float(sk_model.intercept_[0])
    labels = ["Intercept"] + [pretty(n) for n in top.index] + [f"{len(contrib) - len(top)} other features", "Score"]
    values = [intercept] + top.tolist() + [rest, 0]
    fig = go.Figure(go.Waterfall(
        orientation="h", y=labels, x=values, measure=["absolute"] + ["relative"] * (len(top) + 1) + ["total"],
        increasing=dict(marker=dict(color=CHURN)), decreasing=dict(marker=dict(color=STAY)),
        totals=dict(marker=dict(color=INK_2)), connector=dict(line=dict(color=RULE, width=1)),
        text=[f"{v:+.2f}" for v in values[:-1]] + [f"{intercept + contrib.sum():+.2f}"], textposition="outside",
        textfont=dict(size=11, color=INK_2), cliponaxis=False,
        hovertemplate="%{y}<br>%{x:+.3f} log-odds<extra></extra>"))
    fig.update_yaxes(autorange="reversed", showgrid=False, tickfont=dict(color=INK_2, size=12), ticksuffix="  ")
    running = np.cumsum([intercept] + top.tolist() + [rest])
    lo, hi = min(0, running.min()), max(0, running.max())
    pad = (hi - lo) * 0.18 + 0.1
    fig.update_xaxes(showline=False, range=[lo - pad, hi + pad])
    fig.add_vline(x=0, line=dict(color=RULE, width=1))
    fig.update_layout(waterfallgap=0.45)
    style(fig, "Log-odds of leaving", None, height=400)
    return card, fig

## Layout and launch

In [9]:
theme = gr.themes.Base(
    primary_hue=gr.themes.colors.blue, neutral_hue=gr.themes.colors.gray,
    font=["-apple-system", "BlinkMacSystemFont", "Segoe UI", "Roboto", "Helvetica", "Arial", "sans-serif"],
    font_mono=["ui-monospace", "SFMono-Regular", "Menlo", "monospace"],
    radius_size=gr.themes.sizes.radius_sm,
).set(
    body_background_fill=PAGE, body_text_color=INK, body_text_color_subdued=MUTED,
    background_fill_primary=PAGE, background_fill_secondary=PAGE,
    block_background_fill=PAGE, block_border_width="0px", block_shadow="none",
    block_label_text_color=MUTED, block_title_text_color=INK_2, block_title_text_weight="500",
    border_color_primary=RULE, input_background_fill=PAGE, input_border_color=RULE, input_shadow="none",
    slider_color=INK, color_accent=INK, color_accent_soft=GRID, layout_gap="12px",
)

# Gradio adds `.dark` to <body> when the operating system is in dark mode; the page is designed light only,
# so the class is removed on load and whenever Gradio adds it back.
FORCE_LIGHT = """
() => {
  const light = () => { document.body.classList.remove('dark'); document.documentElement.classList.remove('dark'); };
  light();
  new MutationObserver(light).observe(document.body, {attributes: true, attributeFilter: ['class']});
}
"""

CSS = f"""
:root, body, .dark {{ color-scheme: light; }}
body, gradio-app, .gradio-container {{ background: {PAGE} !important; }}
.gradio-container {{ max-width: 1120px !important; margin: 0 auto !important; padding: 0 24px 64px !important; }}
.gradio-container * {{ font-variant-numeric: tabular-nums; }}
footer, .modebar-container {{ display: none !important; }}
.chart, .chart > * {{ border: 0 !important; background: transparent !important; box-shadow: none !important; }}

.page-head {{ padding: 36px 0 20px; }}
.page-head h1 {{ font-size: 28px; font-weight: 650; letter-spacing: -0.01em; color: {INK}; margin: 0 0 10px; }}
.page-head p {{ font-size: 15px; line-height: 1.6; color: {INK_2}; max-width: 70ch; margin: 0 0 8px; }}
.page-head .meta {{ font-size: 13px; color: {MUTED}; margin: 0; }}

/* tabs: plain text with an underline */
.tab-wrapper {{ border-bottom: 1px solid {RULE}; margin-bottom: 8px; }}
.tab-wrapper button {{ font-size: 14px !important; color: {MUTED} !important; padding: 10px 0 !important;
                       margin-right: 24px !important; }}
.tab-wrapper button.selected {{ color: {INK} !important; font-weight: 600; }}

h3.h {{ font-size: 15px; font-weight: 600; color: {INK}; margin: 28px 0 2px; }}
p.cap {{ font-size: 13px; line-height: 1.55; color: {MUTED}; margin: 2px 0 6px; max-width: 80ch; }}
p.cap b {{ color: {INK_2}; font-weight: 600; }}

/* series names in text, replacing chart legends */
.key {{ color: {INK_2}; font-weight: 500; white-space: nowrap; }}
.key::before {{ content: ""; display: inline-block; width: 9px; height: 9px; margin: 0 5px 0 1px;
                border-radius: 2px; background: var(--c); vertical-align: 0; }}
.key.point::before {{ border-radius: 50%; }}
.key.line::before, .key.dash::before, .key.dot::before {{ width: 16px; height: 2.5px; vertical-align: 3px;
                                                         border-radius: 0; }}
.key.dash::before {{ background: repeating-linear-gradient(90deg, var(--c) 0 5px, transparent 5px 8px); }}
.key.dot::before {{ background: repeating-linear-gradient(90deg, var(--c) 0 2px, transparent 2px 4px); }}

.stats {{ display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 20px; }}
.stats.three {{ grid-template-columns: 1fr; gap: 14px; margin-top: 10px; }}
.stats b {{ display: block; font-size: 26px; font-weight: 600; color: {INK}; line-height: 1.2; }}
.stats span {{ font-size: 13px; color: {MUTED}; }}

.cm {{ display: grid; grid-template-columns: auto 1fr 1fr; gap: 6px; margin: 8px 0 4px; }}
.cm .colhead {{ font-size: 12px; color: {MUTED}; padding: 0 2px; }}
.cm .rowhead {{ font-size: 13px; color: {INK_2}; align-self: center; padding-right: 10px; }}
.cm .cell {{ background: #f5f5f4; border-radius: 6px; padding: 12px 14px; }}
.cm .cell.wrong {{ background: #fcefe9; }}
.cm .cell b {{ display: block; font-size: 22px; font-weight: 600; color: {INK}; }}
.cm .cell span {{ font-size: 12px; color: {MUTED}; }}

.tbl {{ width: 100%; border-collapse: collapse; font-size: 13.5px; color: {INK}; margin: 8px 0 6px; }}
.tbl, .tbl tr, .tbl th, .tbl td {{ border: 0 !important; background: none !important; }}
.tbl th {{ font-weight: 500; color: {MUTED}; font-size: 12.5px; text-align: left; padding: 6px 10px;
           border-bottom: 1px solid {RULE} !important; }}
.tbl td {{ padding: 7px 10px; border-bottom: 1px solid {GRID} !important; }}
.tbl td:first-child, .tbl th:first-child {{ padding-left: 0; white-space: nowrap; }}
.tbl th:not(:first-child), .tbl td:not(:first-child) {{ text-align: right; }}
.tbl.cmp tr:nth-child(3) td {{ border-bottom: 1px solid {RULE} !important; }}
.tbl.cmp tr:nth-child(n+4) td {{ color: {INK_2}; }}

.score .big {{ font-size: 44px; font-weight: 650; color: {INK}; margin: 6px 0 0; line-height: 1.1; }}
.score p {{ font-size: 14px; color: {INK_2}; line-height: 1.55; margin: 4px 0 10px; }}

/* feature picker: a row of text options, the selected one underlined */
.pick .wrap {{ gap: 2px 20px !important; }}
.pick label {{ background: none !important; border: 0 !important; box-shadow: none !important;
               border-radius: 0 !important; padding: 4px 0 !important; color: {MUTED} !important;
               border-bottom: 2px solid transparent !important; font-size: 13.5px !important; }}
.pick label.selected {{ color: {INK} !important; border-bottom-color: {INK} !important; }}
.pick input[type=radio] {{ display: none !important; }}

/* sliders: no reset button, a quiet number */
.reset-button {{ display: none !important; }}
input[type=number] {{ border: 0 !important; box-shadow: none !important; text-align: right; width: 4.5em !important;
                      font-weight: 600; color: {INK} !important; background: none !important; }}

@media (max-width: 760px) {{
  .stats {{ grid-template-columns: repeat(2, minmax(0, 1fr)); }}
  .tbl {{ font-size: 12.5px; }} .tbl th, .tbl td {{ padding: 6px; }}
  .table-scroll {{ overflow-x: auto; }}
}}
"""


def heading(title: str, caption: str = "") -> gr.HTML:
    return gr.HTML(f'<h3 class="h">{title}</h3>' + (f'<p class="cap">{caption}</p>' if caption else ""),
                   padding=False)


def chart(fig: go.Figure) -> gr.Plot:
    return gr.Plot(fig, show_label=False, container=False, elem_classes="chart")


with gr.Blocks(title="Undecided PSOE voters") as demo:
    gr.HTML(header_html, padding=False)

    with gr.Tabs():
        with gr.Tab("Threshold"):
            s0, e0, c0, m0, h0 = decide(T_VAL)
            heading("Where to draw the line",
                    f"A voter whose probability of leaving is above the threshold is counted as leaving PSOE. "
                    f"Scored on the {len(Y_TEST):,} declared voters of the test months (May to September 2026), "
                    f"allocated as if they were undecided. The threshold picked on the validation months is "
                    f"{T_VAL:.2f}.")
            with gr.Row(equal_height=False):
                with gr.Column(scale=2, min_width=260):
                    thr = gr.Slider(0.01, 0.99, value=T_VAL, step=0.01, label="Threshold")
                with gr.Column(scale=5):
                    summary = gr.HTML(s0, padding=False)
            with gr.Row(equal_height=False):
                with gr.Column(scale=3):
                    heading("Monthly error by threshold",
                            "How far the estimated churn rate is from the real one, averaged over the four test "
                            "months. Dotted lines: estimates that don't use a threshold.")
                    error_plot = chart(e0)
                with gr.Column(scale=2):
                    heading("Confusion matrix")
                    cm = gr.HTML(c0, padding=False)
            with gr.Row(equal_height=False):
                with gr.Column():
                    heading("Churn by month",
                            f"{key('actual', INK, 'line')}, {key('model', BLUE, 'point')} and "
                            f"{key('closest-party rule', FAINT, 'point')}.")
                    month_plot = chart(m0)
                with gr.Column():
                    heading("Predicted probability",
                            f"Voters who {key('stayed', STAY)} and who {key('left', CHURN)}. "
                            "The black line is the threshold.")
                    split_plot = chart(h0)
            thr.change(decide, thr, [summary, error_plot, cm, month_plot, split_plot], show_progress="hidden")

        with gr.Tab("Three implementations"):
            heading("One model, trained three ways",
                    "Same features, same split, same L2 penalty. Test months; thresholds picked on validation. Churn rates and "
                    "errors use the corrected survey weights.")
            gr.HTML(f'<div class="table-scroll">{comparison_html()}</div>{comparison_caption()}', padding=False)
            with gr.Row(equal_height=False):
                with gr.Column():
                    heading("Predicted vs actual",
                            f"Test voters in ten groups by predicted probability: {key('scikit-learn', BLUE, 'line')}, "
                            f"{key('manual loop', ORANGE, 'dash')}, {key('nn.Module', AQUA, 'dot')}. "
                            "The three lines overlap; dotted is perfect calibration.")
                    chart(calibration_fig())
                with gr.Column():
                    heading("Where the coefficients differ",
                            "PyTorch minus scikit-learn, in units of 10<sup>−5</sup>, added up within each one-hot "
                            "block. Each full one-hot block sums to a column of ones, like the intercept, so only "
                            "the small L2 penalty decides how weight is shared between them. Predictions don't "
                            "move.")
                    chart(coef_gap_fig())
            with gr.Row(equal_height=False):
                with gr.Column():
                    hp = metrics["hyperparameters"]
                    heading("Convergence",
                            f"Training objective minus scikit-learn's, log scale: {key('manual loop', ORANGE, 'dash')} "
                            f"and {key('nn.Module', AQUA, 'dot')}. Learning rate {hp['learning_rate']:g}, "
                            f"{hp['epochs']:,} full-batch epochs. Below about 10<sup>−12</sup> the gap is rounding noise.")
                    chart(convergence_fig())
                with gr.Column():
                    heading("By test month", "Estimated churn rate from each method.")
                    gr.HTML(months_html(), padding=False)

        with gr.Tab("Data"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=3):
                    heading("Correcting the sample",
                            f"Share of each barometer that recalls voting PSOE in 2023, with the {key('CIS weight', CHURN, 'line')}. "
                            "Every barometer is re-weighted on its whole sample to the 23J result, the census (sex × age) "
                            "and education, before the PSOE voters are kept. All weighted figures here use that "
                            "correction.")
                    chart(correction_fig())
                with gr.Column(scale=2):
                    heading("Averages over the 16 barometers")
                    gr.HTML(correction_html(), padding=False)
            with gr.Row(equal_height=False):
                with gr.Column(scale=3):
                    heading("Churn and undecided, by month",
                            "2023 PSOE voters, weighted. Shaded: validation and test months.")
                    chart(over_time_fig())
                with gr.Column(scale=2):
                    heading("Where leavers go", "Declared intention of the 2023 PSOE voters who left, weighted.")
                    chart(destination_fig())
            heading("Churn by feature", "Declared voters only: there is no target for the undecided.")
            drv = gr.Radio(list(DRIVERS), value="Rating of Sánchez", show_label=False, container=False,
                           elem_classes="pick")
            r0, d0 = driver_figs("Rating of Sánchez")
            with gr.Row(equal_height=False):
                with gr.Column():
                    heading("Share who leave")
                    rate_plot = chart(r0)
                with gr.Column():
                    heading("Distribution", f"Within voters who {key('stayed', STAY)} and who {key('left', CHURN)}.")
                    dist_plot = chart(d0)
            drv.change(driver_figs, drv, [rate_plot, dist_plot], show_progress="hidden")
            with gr.Row(equal_height=False):
                with gr.Column(scale=3):
                    heading("PSOE retention, three ways of allocating the undecided",
                            f"{key('model', BLUE, 'line')}, {key('closest-party rule', INK_2, 'dash')}, "
                            f"{key('all go back', FAINT, 'dot')}. Declared voters as they answered. Weighted.")
                    chart(allocation_fig())
                with gr.Column(scale=2):
                    heading("Undecided vs answered",
                            "Average of the model's inputs. The model is trained on the first column and used "
                            "on the second.")
                    gr.HTML(undecided_html(), padding=False)

        with gr.Tab("One voter"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=2):
                    heading("A hypothetical undecided voter")
                    w_closest = gr.Dropdown(NOMINAL_INPUTS["CERCANIA"], value="PSOE",
                                            label="Party closest to their ideas")
                    with gr.Row():
                        w_sanchez = gr.Slider(1, 10, value=5, step=1, label="Rates Sánchez")
                        w_feijoo = gr.Slider(1, 10, value=3, step=1, label="Rates Feijóo")
                    with gr.Row():
                        w_diaz = gr.Slider(1, 10, value=5, step=1, label="Rates Díaz")
                        w_abascal = gr.Slider(1, 10, value=1, step=1, label="Rates Abascal")
                    with gr.Row():
                        w_probvoto = gr.Slider(0, 10, value=8, step=1, label="Probability of voting")
                        w_ideology = gr.Slider(1, 10, value=4, step=1, label="Ideology (1 left, 10 right)")
                    with gr.Row():
                        w_eco_es = gr.Dropdown(list(ECO), value="Bad", label="Spain's economy")
                        w_eco_per = gr.Dropdown(list(ECO), value="Good", label="Personal economy")
                    w_p1 = gr.Dropdown(list(PROBLEMS), value="La vivienda", label="Spain's main problem")
                    with gr.Accordion("Other problems and demographics", open=False):
                        with gr.Row():
                            w_p2 = gr.Dropdown(list(PROBLEMS), value="(none)", label="Second problem")
                            w_p3 = gr.Dropdown(list(PROBLEMS), value="(none)", label="Third problem")
                        with gr.Row():
                            w_age = gr.Slider(18, 95, value=50, step=1, label="Age")
                            w_sex = gr.Radio(NOMINAL_INPUTS["SEXO"], value=mode_of["SEXO"], label="Sex")
                        with gr.Row():
                            w_edu = gr.Dropdown(NOMINAL_INPUTS["ESTUDIOS"], value=mode_of["ESTUDIOS"],
                                                label="Education")
                            w_rel = gr.Dropdown(NOMINAL_INPUTS["RELIGION"], value=mode_of["RELIGION"],
                                                label="Religion")
                        with gr.Row():
                            w_work = gr.Dropdown(NOMINAL_INPUTS["SITLAB"], value=mode_of["SITLAB"],
                                                 label="Work status")
                            w_class = gr.Dropdown(NOMINAL_INPUTS["CLASESOCIAL"], value=mode_of["CLASESOCIAL"],
                                                  label="Social class")
                        with gr.Row():
                            w_region = gr.Dropdown(NOMINAL_INPUTS["CCAA"], value=mode_of["CCAA"], label="Region")
                            w_marital = gr.Dropdown(NOMINAL_INPUTS["ECIVIL"], value=mode_of["ECIVIL"],
                                                    label="Marital status")
                        with gr.Row():
                            w_income = gr.Dropdown(list(INCOME), value=list(INCOME)[3],
                                                   label="Household income (net per month)")
                            w_town = gr.Dropdown(list(TOWN), value=list(TOWN)[4], label="Town size")
                w_inputs = [w_closest, w_sanchez, w_feijoo, w_diaz, w_abascal, w_probvoto, w_ideology, w_eco_es,
                            w_eco_per, w_p1, w_p2, w_p3, w_age, w_sex, w_edu, w_rel, w_work, w_class, w_region,
                            w_marital, w_income, w_town]
                card0, fig0 = score_voter(*[c.value for c in w_inputs])
                with gr.Column(scale=1, min_width=280):
                    heading("Result")
                    w_card = gr.HTML(card0, padding=False)
            heading("Why this score",
                    f"Each bar is a scikit-learn coefficient times this voter's standardised answer. Bars that "
                    f"push {key('towards leaving', CHURN)} and {key('towards staying', STAY)} add up to the score.")
            w_plot = chart(fig0)
            for comp in w_inputs:
                comp.change(score_voter, w_inputs, [w_card, w_plot], show_progress="hidden")

demo.launch(theme=theme, css=CSS, js=FORCE_LIGHT, inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
